# AutoSort Training Pipeline

Main steps:
1. Threshold detection
2. Training data preparation
3. Model training


In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre

from pathlib import Path
from utils_clean import (
    prepare_training_data,
    train_autosort_model
)


In [4]:
# Load data
recording_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5"
spike_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique/spike_inf.tsv"
neuron_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique/neuron_inf.pkl"

# Load GT data
spike_inf = pd.read_csv(spike_inf_path, sep='\t', index_col=0)
with open(neuron_inf_path, 'rb') as f:
    neuron_inf = pickle.load(f)

recording, sorting = se.read_mearec(recording_path)
#recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
recording_f = spre.bandpass_filter(recording, freq_min=300, freq_max=3000)
recording_f = spre.common_reference(recording_f, reference="global", operator="median")

print(f"Recording loaded successfully")
print(f"Sampling rate: {recording_f.get_sampling_frequency()} Hz")
print(f"Number of channels: {recording_f.get_num_channels()}")


Recording loaded successfully
Sampling rate: 10000.0 Hz
Number of channels: 384


## Step 1: Threshold Detection + Training Data Preparation


In [15]:
# Set parameters
save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/test/"
duration_seconds = 50  # Processing duration (seconds)

# Extract all unique tract_channels from neuron_inf for threshold detection on these channels only
valid_channels = sorted(neuron_inf['tract_channel'].unique().tolist())
print(f"Number of valid channels extracted from neuron_inf: {len(valid_channels)}")
print(f"Valid channels list: {valid_channels}")

# Detection parameters (consistent with AutoSort default values)
detection_params = {
    'thr_min': 2.2,
    'thr_max': 15,
    'distance': 3,
    'ch_max_simul_firing': 8,
    'wlen': 10,
    'prominence': 5,
}

# Waveform window parameters
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# Prepare training data (includes threshold detection, GT matching, waveform extraction, data saving)
train_data_dir = prepare_training_data(
    recording_f=recording_f,
    spike_inf=spike_inf,
    neuron_inf=neuron_inf,
    save_dir=save_dir,
    duration_seconds=duration_seconds,
    valid_channels=valid_channels,  # Pass valid_channels parameter to detect only on valid channels
    **detection_params,
    **window_params
)


Number of valid channels extracted from neuron_inf: 131
Valid channels list: [0, 2, 4, 5, 6, 9, 12, 15, 16, 17, 19, 21, 25, 26, 28, 30, 32, 33, 34, 36, 41, 44, 45, 48, 50, 51, 53, 54, 58, 60, 62, 72, 73, 74, 77, 78, 79, 82, 83, 84, 86, 89, 94, 95, 100, 106, 107, 110, 113, 115, 118, 119, 122, 127, 138, 139, 141, 145, 146, 150, 152, 160, 161, 163, 166, 170, 171, 172, 174, 185, 188, 189, 196, 197, 199, 212, 218, 220, 221, 222, 229, 232, 234, 237, 238, 242, 243, 251, 261, 262, 264, 273, 274, 276, 279, 291, 292, 294, 296, 299, 300, 301, 311, 312, 313, 320, 322, 324, 334, 338, 339, 342, 344, 346, 348, 349, 353, 356, 359, 362, 364, 367, 368, 370, 371, 374, 375, 377, 380, 381, 382]
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 384
Recording total length: 36000000 samples (3600.00 seconds)
Will process first 500000 samples (50.00 seconds)
Number of valid channels: 131
Valid channels list: [0, 2, 4, 5, 6, 9, 12, 15, 16, 17, 19, 21, 25, 26, 28, 30, 32, 33, 34, 36, 41, 

Extracting waveforms:   7%|▋         | 2/30 [00:11<02:46,  5.95s/it]


KeyboardInterrupt: 

## Step 2: Model Training


In [4]:
# Set training parameters
base_model_save_dir = save_dir + "model_save/"
n_channels = recording_f.get_num_channels()

training_params = {
    'epochs': 20,
    'batch_size': 512,
    'left_sample': 10,
    'right_sample': 20,
    'early_stopping': True,  # Enable early stopping
    'patience': 5,  # Stop if accuracy doesn't improve for 5 consecutive epochs
    'min_delta': 0.0,  # Minimum change
}

# Repeat training 5 times
n_runs = 5
all_models = []
all_logs = []

for run_id in range(1, n_runs + 1):
    print(f"\n{'='*60}")
    print(f"Starting training run {run_id}/{n_runs}")
    print(f"{'='*60}")
    
    # Create independent save directory for each training run
    model_save_dir = base_model_save_dir + f"run_{run_id}/"
    
    # Train model
    autosort_model, training_log = train_autosort_model(
        train_data_dir=train_data_dir,
        model_save_dir=model_save_dir,
        n_channels=n_channels,
        **training_params
    )
    
    all_models.append(autosort_model)
    all_logs.append(training_log)
    
    print(f"\nTraining run {run_id} completed!")
    print(f"Model save directory: {model_save_dir}")

print(f"\n{'='*60}")
print(f"All {n_runs} training runs completed!")
print(f"{'='*60}")



开始第 1/5 次训练
使用设备: cuda
创建 dataset...
Dataset 加载完成:
  - 总样本数: 431525
  - 通道数: 30
  - 窗口长度: 30
  - 唯一单元数: 27
  - 噪声样本数: 362227.0
  - 非噪声样本数: 69298.0
模型参数:
  - 通道数: 30
  - 窗口长度: 30
  - 单元数量: 27
  - 输入维度: 930
单元ID列表已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/model_save/run_1/keep_id.pkl

数据集划分:
  - 训练集: 345220 样本
  - 验证集: 86305 样本
已加载现有模型

第 1 次训练完成！
模型保存目录: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/model_save/run_1/

开始第 2/5 次训练
使用设备: cuda
创建 dataset...
Dataset 加载完成:
  - 总样本数: 431525
  - 通道数: 30
  - 窗口长度: 30
  - 唯一单元数: 27
  - 噪声样本数: 362227.0
  - 非噪声样本数: 69298.0
模型参数:
  - 通道数: 30
  - 窗口长度: 30
  - 单元数量: 27
  - 输入维度: 930
单元ID列表已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/model_save/run_2/keep_id.pkl

数据集划分:
  - 训练集: 345220 样本
  - 验证集: 86305 样本
已加载现有模型

第 2 次训练完成！
模型保存目录: /media/ubuntu/s